# Data Migration Demo

Data migration from Google Sheets to Airtable using
* gspread
* pyairtable

In [18]:
# Import our deps
from dotenv import load_dotenv
import gspread
from pyairtable import Api
import os

In [37]:
# Load our secret keys / environment variables
load_dotenv()

dataset = "spotify_alltime_top100_songs"

In [38]:
# Authenticate gspread
gc = gspread.service_account()

# Open sheet
spread_sheet_name = "tech-ops-test-sheet"
work_sheet_name = dataset

sh = gc.open(spread_sheet_name)
work_sheet = sh.worksheet(work_sheet_name)

# Test connection
print(work_sheet.get('A1'))

[['alltime_rank']]


In [39]:
# Authenticate pyairtable
api = Api(os.environ['AIRTABLE_ACCESS_TOKEN'])
base_id = os.environ['AIRTABLE_BASE_ID']


table_name = dataset
base = api.base(base_id)
table_exists = False

for table in base.tables():
    if table_name == table.name:
        table_exists = True

In [ ]:
# Migrate dataset to airtable if the table doesn't exist
if table_exists:
    raise SystemExit("Stop here: Table already exists")


# Create an Airtable schema for our dataset
fields = [
    {"name": "alltime_rank", "type": "number", "options": {"precision": 0}},
    {"name": "song_title", "type": "singleLineText"},
    {"name": "artist", "type": "singleLineText"},
    {"name": "total_streams_billions", "type": "number", "options": {"precision": 2}},
    {"name": "primary_genre", "type": "singleLineText"},
    {"name": "bpm", "type": "number", "options": {"precision": 0}},
    {"name": "release_year", "type": "number", "options": {"precision": 0}},
    {"name": "artist_country", "type": "singleLineText"},
    {"name": "explicit", "type": "checkbox", "options": {"icon": "check", "color": "redBright"}},
    {"name": "danceability", "type": "number", "options": {"precision": 2}},
    {"name": "energy", "type": "number", "options": {"precision": 2}},
    {"name": "valence", "type": "number", "options": {"precision": 2}},
    {"name": "acousticness", "type": "number", "options": {"precision": 2}},
    {"name": "dataset_part", "type": "singleLineText"},
]

table = base.create_table(name=table_name, fields=fields)

# Since we know our dataset only has 100 rows we'll just grab them all at once
records = work_sheet.get_all_records()

# Clean the 'explicit' column to be python True False instead of string
for record in records:
    record['explicit'] = True if record['explicit'] == 'TRUE'else False

# Migrate!
table.batch_create(records)
    

[{'id': 'recptrKqJTozFaYvd',
  'createdTime': '2026-03-15T10:59:00.000Z',
  'fields': {'alltime_rank': 1,
   'song_title': 'Blinding Lights',
   'artist': 'The Weeknd',
   'total_streams_billions': 5.26,
   'primary_genre': 'Synth-Pop',
   'bpm': 171,
   'release_year': 2019,
   'artist_country': 'Canada',
   'explicit': True,
   'danceability': 0.51,
   'energy': 0.8,
   'valence': 0.33,
   'acousticness': 0,
   'dataset_part': 'Spotify All-Time Most Streamed Top 100'}},
 {'id': 'recdZniw8cCjTKQoR',
  'createdTime': '2026-03-15T10:59:00.000Z',
  'fields': {'alltime_rank': 2,
   'song_title': 'Shape of You',
   'artist': 'Ed Sheeran',
   'total_streams_billions': 4.9,
   'primary_genre': 'Pop/Dancehall',
   'bpm': 96,
   'release_year': 2017,
   'artist_country': 'UK',
   'explicit': True,
   'danceability': 0.83,
   'energy': 0.65,
   'valence': 0.93,
   'acousticness': 0.08,
   'dataset_part': 'Spotify All-Time Most Streamed Top 100'}},
 {'id': 'recnonEFGGKPycqkI',
  'createdTime': '